# OEC Whisper Turbo — Gradio speech-to-text worker

Inference is kept in an isolated provider backend. The phase2_3d notebook kernel is a Gradio client only and does not import that model's ML stack.

## Environment and startup
- UI kernel: phase2_3d (frontend only; existing Gradio environment).
- Backend: ~/.local/venvs/oec-audio-py311/bin/python (isolated speech runtime).
- API: http://127.0.0.1:7872. Require /health to return ready=true and model_loaded=true.
- No installs or downloads are performed by this notebook. Check nvidia-smi, python storage_manager.py --status, and provider service status before launch.
- API binds to 127.0.0.1 only. Keep share=False; use an approved SSH tunnel for remote use.

From the OEC repository root, after confirming a genuinely free A100 and a permitted storage state:

    python tools/oec_service_manager.py status audio-api
    python tools/oec_service_manager.py start audio-api --provider whisper --gpu 1
    python tools/oec_service_manager.py status audio-api
    python tools/oec_service_manager.py logs audio-api --tail 80

The GPU index is an example, not a reservation; these provider starts are alternatives, not a batch. Never kill an unrelated process or bypass StorageManager admission.

Accepts WAV/MP3/FLAC/OGG/M4A, at most 12 MiB. One successful transcript is not a broad WER evaluation.

In [ ]:
from pathlib import Path
import base64
import json
import os
import tempfile
import time
import gradio as gr
import requests

API_BASE = os.environ.get("OEC_MEDIA_API_URL", "http://127.0.0.1:7872").rstrip("/")
RESULT_ROOT = Path(tempfile.gettempdir()) / "oec-notebook-results"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

def service_health():
    try:
        response = requests.get(f"{API_BASE}/health", timeout=8)
        payload = response.json()
    except Exception as exc:
        raise gr.Error(f"Service unreachable at {API_BASE}: {type(exc).__name__}. Start the approved whisper service first.") from exc
    if response.status_code != 200 or not payload.get("ready") or not payload.get("model_loaded"):
        raise gr.Error(f"whisper is not ready: {json.dumps(payload, ensure_ascii=False)}")
    return payload

def submit_and_wait(payload, timeout_seconds=1800):
    health = service_health()
    response = requests.post(f"{API_BASE}/v1/jobs", json=payload, timeout=30)
    if response.status_code != 202:
        raise gr.Error(f"Job submission refused (HTTP {response.status_code}): {response.text[:1200]}")
    accepted = response.json()
    job_id = accepted.get("job_id")
    if not isinstance(job_id, str) or not job_id.startswith("job_"):
        raise gr.Error(f"Unexpected job response: {accepted}")
    deadline = time.monotonic() + timeout_seconds
    latest = accepted
    while time.monotonic() < deadline:
        poll = requests.get(f"{API_BASE}/v1/jobs/{job_id}", timeout=20)
        if poll.status_code != 200:
            raise gr.Error(f"Job status lookup failed (HTTP {poll.status_code}): {poll.text[:800]}")
        latest = poll.json()
        status = latest.get("status")
        if status == "succeeded":
            break
        if status in {"failed", "error", "cancelled"}:
            raise gr.Error(f"whisper job {status}: {latest.get('message', 'no error details returned')}")
        time.sleep(2)
    else:
        raise gr.Error(f"whisper job timed out; inspect the service-manager logs.")
    result = requests.get(f"{API_BASE}/v1/jobs/{job_id}/result", timeout=120)
    if result.status_code != 200:
        raise gr.Error(f"Result download failed (HTTP {result.status_code}): {result.text[:800]}")
    content_type = result.headers.get("Content-Type", "")
    if content_type.startswith("text/plain"):
        output = result.text.strip()
    else:
        output_path = RESULT_ROOT / f"{job_id}.txt"
        output_path.write_bytes(result.content)
        if not output_path.is_file() or output_path.stat().st_size == 0:
            raise gr.Error("Result file is missing or empty.")
        output = str(output_path)
    details = {"job_id": job_id, "status": latest.get("status"), "api": API_BASE,
               "health": health, "metadata": latest.get("metadata", {})}
    return output, json.dumps(details, indent=2, ensure_ascii=False)


In [ ]:
def transcribe_audio(audio_path, language, initial_prompt):
    if not audio_path:
        raise gr.Error("Upload or record an audio file.")
    raw = Path(audio_path).read_bytes()
    if not raw or len(raw) > 12 * 1024 * 1024:
        raise gr.Error("Audio must be non-empty and at most 12 MiB.")
    payload = {"task": "transcribe",
               "parameters": {"language": None if language == "auto" else language,
                              "initial_prompt": initial_prompt or None,
                              "temperature": 0.0, "condition_on_previous_text": True},
               "audio_base64": base64.b64encode(raw).decode("ascii")}
    return submit_and_wait(payload, timeout_seconds=600)

demo = gr.Interface(
    fn=transcribe_audio,
    inputs=[gr.Audio(type="filepath", label="Audio file (WAV/MP3/FLAC/OGG/M4A)"), gr.Dropdown(["auto", "en", "hi", "zh", "ja", "fr", "de", "es"], value="auto", label="Language"), gr.Textbox(label="Optional initial prompt", lines=2)],
    outputs=[gr.Textbox(label="Transcript", lines=8), gr.Textbox(label="Job metadata", lines=8)],
    title="Whisper Turbo — Gradio speech-to-text worker",
    description="Backend remains isolated; this notebook submits jobs and retrieves validated results."
)
if os.environ.get("OEC_GRADIO_HEADLESS_SMOKE") != "1":
    demo.queue(max_size=1).launch(server_name="127.0.0.1", server_port=7905, share=False, inline=False)
